In [1]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName('NetworkIntrusionETL_Extract') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'Data')).replace('\\', '/')
STAGING  = os.path.abspath(os.path.join(os.getcwd(), '..', 'Data', 'staging')).replace('\\', '/')
print('Data dir   :', DATA_DIR)
print('Staging dir:', STAGING)

Spark version: 4.1.1
Data dir   : d:/Projects/Big Data/BG  Assignmnet/Assignmnet/Data
Staging dir: d:/Projects/Big Data/BG  Assignmnet/Assignmnet/Data/staging


In [2]:
# ── Source 1: CICIDS2017 CSV ──────────────────────────────────────────────────
cicids_df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(DATA_DIR + '/cicids2017_cleaned.csv')

print('CICIDS2017 rows:', cicids_df.count())
print('CICIDS2017 columns:', len(cicids_df.columns))
cicids_df.printSchema()

CICIDS2017 rows: 2520751
CICIDS2017 columns: 53
root
 |-- Destination Port: integer (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |-- Flow Packets/s: double (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow IAT Std: double (nullable = true)
 |-- Flow IAT Max: integer (nullable = true)
 |-- Flow IAT Min: integer (nullable = true)
 |-- Fwd IAT Total: integer (nullable = tru

In [3]:
# ── Source 2: UNSW-NB15 Training Parquet ─────────────────────────────────────
unsw_train_df = spark.read.parquet(DATA_DIR + '/UNSW_NB15_training-set.parquet')

print('UNSW Train rows:', unsw_train_df.count())
print('UNSW Train columns:', len(unsw_train_df.columns))
unsw_train_df.printSchema()

UNSW Train rows: 175341
UNSW Train columns: 36
root
 |-- dur: float (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- state: string (nullable = true)
 |-- spkts: short (nullable = true)
 |-- dpkts: short (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- rate: float (nullable = true)
 |-- sload: float (nullable = true)
 |-- dload: float (nullable = true)
 |-- sloss: short (nullable = true)
 |-- dloss: short (nullable = true)
 |-- sinpkt: float (nullable = true)
 |-- dinpkt: float (nullable = true)
 |-- sjit: float (nullable = true)
 |-- djit: float (nullable = true)
 |-- swin: short (nullable = true)
 |-- stcpb: long (nullable = true)
 |-- dtcpb: long (nullable = true)
 |-- dwin: short (nullable = true)
 |-- tcprtt: float (nullable = true)
 |-- synack: float (nullable = true)
 |-- ackdat: float (nullable = true)
 |-- smean: short (nullable = true)
 |-- dmean: short (nullable = true)
 |-- 

In [4]:
# ── Source 3: Zeek conn.log ───────────────────────────────────────────────────
conn_df = spark.read \
    .option('sep', '\t') \
    .option('comment', '#') \
    .option('header', 'false') \
    .option('inferSchema', 'true') \
    .csv(DATA_DIR + '/conn (1).log')

conn_cols = [
    'ts', 'uid', 'orig_h', 'orig_p', 'resp_h', 'resp_p',
    'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
    'conn_state', 'local_orig', 'missed_bytes', 'history',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'tunnel_parents'
]
conn_df = conn_df.toDF(*conn_cols)

print('Zeek conn.log rows:', conn_df.count())
conn_df.printSchema()

Zeek conn.log rows: 36394
root
 |-- ts: double (nullable = true)
 |-- uid: string (nullable = true)
 |-- orig_h: string (nullable = true)
 |-- orig_p: integer (nullable = true)
 |-- resp_h: string (nullable = true)
 |-- resp_p: integer (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- orig_bytes: string (nullable = true)
 |-- resp_bytes: string (nullable = true)
 |-- conn_state: string (nullable = true)
 |-- local_orig: string (nullable = true)
 |-- missed_bytes: integer (nullable = true)
 |-- history: string (nullable = true)
 |-- orig_pkts: integer (nullable = true)
 |-- orig_ip_bytes: integer (nullable = true)
 |-- resp_pkts: integer (nullable = true)
 |-- resp_ip_bytes: integer (nullable = true)
 |-- tunnel_parents: string (nullable = true)



In [5]:
# ── Write raw staging parquet ─────────────────────────────────────────────────
cicids_df.write.mode('overwrite').parquet(STAGING + '/cicids_raw')
unsw_train_df.write.mode('overwrite').parquet(STAGING + '/unsw_train_raw')
conn_df.write.mode('overwrite').parquet(STAGING + '/conn_log_raw')

print('Extraction complete. Staged to:', STAGING)

Extraction complete. Staged to: d:/Projects/Big Data/BG  Assignmnet/Assignmnet/Data/staging
